# Path C 全套 state-only 重跑（论文主结果 sweep）

| Field | Value |
|---|---|
| Sweep | `configs/sweeps/pathC_full_state_only.yaml` |
| Task | 9 (6 baseline fusion + 3 EFT 改进 C/D/CD) |
| Modality | 4 (V+K+T / V+K / V+T / K+T)，K=km_event、T=telem_60hz 新特征 |
| Split | 2 (cross_subject / within_subject) |
| Seed | 3 (0, 1, 2) |
| **Total runs** | **9 × 4 × 2 × 3 = 216** |
| 目标 GPU | **L4 (24GB)** — 性价比最优 |
| 保守超参 | bs=32, num_workers=4, ckpt_every_batches=500（保 L4 稳定） |
| 估计时长 | **22–29h** on L4 (~6–8 min/run) — Pro+ 24h session 需 1 次重连 |
| runs_root | `runs/pathC_full_state_only/` |

**断点续跑用法**：Colab 断连或主动中断后，重新执行 Cell 3（运行主命令）即可。`run_experiment.py` 的 `find_completed_run()` 会检测 seed 目录下 `metrics.json` 存在并自动跳过，日志开头打印 `Skipped X already-completed runs`。

**F 与 CDF 未纳入**：F 的 `task_aware_multitask_seq` head 依赖 `diff_tasks=[trend]`，state-only 下 trend 移除后 F 改进失效，CDF 退化为 CD。已在 yaml 顶部注释说明，供 paper 讨论。

In [ ]:
# Cell 1: Mount + setup
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyyaml', 'scipy', '-q'])

%cd /content/drive/MyDrive/ProjectExperiment
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Cell 2: Dry run — verify plan = 216 runs, check how many already done
!python -u scripts/run_experiment.py \
    --sweep configs/sweeps/pathC_full_state_only.yaml \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/pathC_full_state_only \
    --dry_run 2>&1 | tail -20

In [ ]:
# Cell 3: MAIN RUN (resume-safe — repeat this cell after any interruption)
# -u: unbuffered stdout for real-time per-epoch progress
# workers=1: single GPU, sequential runs
!python -u scripts/run_experiment.py \
    --sweep configs/sweeps/pathC_full_state_only.yaml \
    --workers 1 \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/pathC_full_state_only

In [ ]:
# Cell 4: Progress check — count done / total per task×split×modality
import glob
from collections import defaultdict

ROOT = '/content/drive/MyDrive/AmuCS_experiment/runs/pathC_full_state_only'
files = glob.glob(f'{ROOT}/**/metrics.json', recursive=True)

counts = defaultdict(int)
for f in files:
    parts = f.replace(ROOT + '/', '').split('/')
    # parts: [split, task_name, modality_combo, seed_dir, metrics.json]
    if len(parts) >= 3:
        key = f'{parts[0]:>14} | {parts[1]:>22} | {parts[2]}'
        counts[key] += 1

TOTAL = 216
done = len(files)
print(f'Completed: {done}/{TOTAL} runs ({100*done/TOTAL:.1f}%)\n')
print(f'{"split":>14} | {"task":>22} | modality                         | done/3')
print('-' * 90)
for key in sorted(counts.keys()):
    n = counts[key]
    mark = '✓' if n == 3 else ' '
    print(f'{key:<66} |  {n}/3 {mark}')

remaining = TOTAL - done
if remaining > 0:
    est_hours = remaining * 7 / 60
    print(f'\nRemaining: {remaining} runs, ~{est_hours:.1f}h at 7min/run on L4.  Re-run Cell 3 to continue.')
else:
    print('\nAll 216 runs complete. Proceed to Cell 5 for results aggregation.')

In [ ]:
# Cell 5: Results aggregation — 3-seed mean±std of test_macro_f1_state per task×modality×split
import json, glob
import numpy as np
from collections import defaultdict

ROOT = '/content/drive/MyDrive/AmuCS_experiment/runs/pathC_full_state_only'
TASKS = ['cma_state_only', 'eft_state_only', 'mft_state_only', 'lft_state_only',
         'late_state_only', 'gated_state_only',
         'eft_C_state_only', 'eft_D_state_only', 'eft_CD_state_only']
MODS = ['triple_video_km_event_telem_60hz', 'dual_video_km_event',
        'dual_video_telem_60hz', 'dual_km_event_telem_60hz']
SPLITS = ['cross_subject', 'within_subject']

def collect(split, task, mod):
    f1, bacc = [], []
    # task dir may have suffix _3seed (auto-added). Use glob with prefix.
    for s in [0, 1, 2]:
        hits = glob.glob(f'{ROOT}/{split}/{task}*/{mod}/*seed{s}*/metrics.json')
        if hits:
            m = json.load(open(sorted(hits)[-1]))
            f1.append(m.get('test_macro_f1_state', m.get('test_macro_f1_mean')))
            bacc.append(m.get('test_balanced_acc_state', m.get('test_balanced_acc_mean')))
    return f1, bacc

for split in SPLITS:
    print('=' * 100)
    print(f'SPLIT = {split}  |  test_macro_f1_state (3-seed mean ± std, ddof=1)')
    print('=' * 100)
    header = f'{"task":<22} | ' + ' | '.join(f'{m[:28]:>28}' for m in MODS)
    print(header)
    print('-' * len(header))
    for task in TASKS:
        row = f'{task:<22} | '
        for mod in MODS:
            f1, _ = collect(split, task, mod)
            if len(f1) == 3:
                m, s = float(np.mean(f1)), float(np.std(f1, ddof=1))
                row += f'{m:.4f}±{s:.4f} ({len(f1)}) | '
            elif len(f1) > 0:
                row += f'{np.mean(f1):.4f} (n={len(f1)})       | '
            else:
                row += f'{"(missing)":>28} | '
        print(row)
    print()

print('\nNote: numbers in parentheses = n seeds. Only n=3 cells are final.')